# 1.10 PyTorch 工程实践：一个"工业级"训练脚本长什么样

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/01_prerequisites/1.10_training_pipeline.ipynb)


## 为什么要学

打开 openpi 的 `scripts/train_pytorch.py`、FluxVLA 的 `fluxvla/engines/runners/`、LeRobot 的 `train.py`，你会发现它们的结构大同小异：

```text
配置 -> 数据集 Dataset -> DataLoader -> 模型 -> 优化器/调度器
     -> 循环 { 取 batch -> 前向 -> 损失 -> 反向 -> 裁剪 -> 更新 -> EMA -> 日志 -> 存档 }
     -> 评估
```

这一课把这些零件逐个讲清楚，并给出一个**你以后可以反复复用的训练模板**。学完这一课，再去读那些大型代码库会轻松很多。

---

## 1. Dataset 与 DataLoader

- `Dataset`：告诉 PyTorch "一共有多少个样本"（`__len__`）和"第 i 个样本是什么"（`__getitem__`）。
- `DataLoader`：负责打乱、分批、多进程加载、把样本拼成 batch（`collate`）。

机器人数据集的特点是：**存的是一条条轨迹，但训练样本是"某条轨迹的某一帧 + 之后 H 步的动作块"**。所以 `__getitem__` 里要做"索引 → (第几条轨迹, 第几帧)"的映射。LeRobot 的 `LeRobotDataset` 用 `delta_timestamps` 实现这个，openpi 在其上套了一层。

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader
from vla_ladder.envs import PointReachEnv, collect_demos

env = PointReachEnv(num_envs=100, seed=0)
episodes = collect_demos(env, n_episodes=200, noise=0.1, seed=0)
print(f"{len(episodes)} 条轨迹，第一条长度 {len(episodes[0]['obs'])}，obs 维度 {episodes[0]['obs'].shape[1]}")

### 练习 1：动作块数据集

实现 `ChunkDataset`：每个样本是 `(obs_t, actions[t:t+H], mask)`。越界的动作用最后一个动作补齐，`mask` 标记哪些是真实动作（`True`）、哪些是补齐的（`False`）。

> ACT 的数据集就返回这样的 `is_pad` 掩码，训练时补齐的位置不计入损失。

In [ ]:
class ChunkDataset(Dataset):
    def __init__(self, episodes, horizon):
        self.episodes, self.H = episodes, horizon
        # index 列表：每个元素是 (轨迹编号, 帧编号)
        self.index = [(e, t) for e, ep in enumerate(episodes) for t in range(len(ep["obs"]))]

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        e, t = self.index[i]
        obs, acts = self.episodes[e]["obs"], self.episodes[e]["actions"]
        T = len(acts)
        # TODO:
        #   idx = t + [0, 1, ..., H-1]，超过 T-1 的部分截断到 T-1
        #   mask = 原始 idx < T
        #   返回 dict(obs=float32 张量 (obs_dim,), actions=(H, act_dim), mask=bool (H,))
        raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
class ChunkDataset(Dataset):
    def __init__(self, episodes, horizon):
        self.episodes, self.H = episodes, horizon
        self.index = [(e, t) for e, ep in enumerate(episodes) for t in range(len(ep["obs"]))]

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        e, t = self.index[i]
        obs, acts = self.episodes[e]["obs"], self.episodes[e]["actions"]
        T = len(acts)
        raw = t + np.arange(self.H)
        idx = np.minimum(raw, T - 1)
        return {
            "obs": torch.as_tensor(obs[t], dtype=torch.float32),
            "actions": torch.as_tensor(acts[idx], dtype=torch.float32),
            "mask": torch.as_tensor(raw < T),
        }

In [ ]:
# ✅ 自测
ds = ChunkDataset(episodes, horizon=8)
s = ds[len(episodes[0]["obs"]) - 2]          # 第 0 条轨迹的倒数第 2 帧
assert s["actions"].shape == (8, 2) and s["mask"].tolist() == [True, True] + [False] * 6
assert torch.allclose(s["actions"][-1], torch.tensor(episodes[0]["actions"][-1]))
loader = DataLoader(ds, batch_size=64, shuffle=True, drop_last=True)
batch = next(iter(loader))                   # 默认 collate 会把 dict 的每个字段分别堆叠
print({k: v.shape for k, v in batch.items()})
print("通过 ✅")

> **DataLoader 的常用参数**：`batch_size`、`shuffle`（训练 True、评估 False）、`num_workers`（多进程加载，图像数据很有用）、`drop_last`、`pin_memory`（加速拷贝到 GPU）、`collate_fn`（自定义拼 batch 的方式）。

---

## 2. 保存与加载：`state_dict`

`model.state_dict()` 是一个"参数名 → 张量"的字典。**存档要同时保存模型、优化器、调度器状态和步数**，否则无法完美续训。

In [ ]:
model = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, 2))
print(list(model.state_dict().keys()))

opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
ckpt = {"model": model.state_dict(), "optimizer": opt.state_dict(), "step": 1234}
torch.save(ckpt, "ckpt_demo.pt")

model2 = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, 2))
opt2 = torch.optim.AdamW(model2.parameters(), lr=1e-3)
loaded = torch.load("ckpt_demo.pt", map_location="cpu")   # map_location：GPU 上存的也能在 CPU 上读
model2.load_state_dict(loaded["model"]); opt2.load_state_dict(loaded["optimizer"])
print("续训从第", loaded["step"], "步开始")

> 加载预训练权重到一个"改过结构"的模型时，用 `model.load_state_dict(sd, strict=False)`，它会返回 `missing_keys` 和 `unexpected_keys`。**一定要打印出来看**，否则可能以为加载成功了，实际上一半参数还是随机的。

---

## 3. EMA：权重的指数滑动平均

训练时维护一份"影子"参数 $\bar\theta \leftarrow \beta\bar\theta + (1-\beta)\theta$，评估和部署时用 $\bar\theta$。它相当于把最近很多步的模型做平均，**更平滑、更鲁棒**。Diffusion Policy 和 openpi 都用它（openpi 默认 `ema_decay=0.99`）。

### 练习 2：实现 EMA

In [ ]:
import copy

class EMA:
    def __init__(self, model, decay=0.99):
        self.decay = decay
        self.shadow = copy.deepcopy(model).eval()      # 影子模型
        for p in self.shadow.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        # TODO: 对每一对 (影子参数 s, 当前参数 p)：s = decay * s + (1 - decay) * p
        #       提示：s.mul_(decay).add_(p, alpha=1 - decay)
        raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
import copy

class EMA:
    def __init__(self, model, decay=0.99):
        self.decay = decay
        self.shadow = copy.deepcopy(model).eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        for s, p in zip(self.shadow.parameters(), model.parameters()):
            s.mul_(self.decay).add_(p.detach(), alpha=1 - self.decay)
        # BatchNorm 的 running_mean 等 buffer 直接复制
        for sb, b in zip(self.shadow.buffers(), model.buffers()):
            sb.copy_(b)

In [ ]:
# ✅ 自测
m = nn.Linear(3, 1); ema = EMA(m, decay=0.9)
with torch.no_grad():
    m.weight.fill_(1.0); m.bias.fill_(1.0)
ema.shadow.weight.data.fill_(0.0); ema.shadow.bias.data.fill_(0.0)
for _ in range(10):
    ema.update(m)
expect = 1 - 0.9 ** 10
assert torch.allclose(ema.shadow.weight, torch.full_like(m.weight, expect), atol=1e-6)
print(f"通过 ✅ 10 次更新后影子权重 = {ema.shadow.weight[0, 0]:.4f}（理论 1-0.9^10 = {expect:.4f}）")

---

## 4. 混合精度训练

`bfloat16` 只占 2 个字节，计算更快、显存减半。π0 的大部分参数都存成 bf16。PyTorch 中用 `autocast` 自动把合适的运算切换到低精度：

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, 2)).to(device)
x = torch.randn(8, 4, device=device)
with torch.autocast(device_type=device, dtype=torch.bfloat16):
    y = model(x)
print("autocast 中输出 dtype:", y.dtype)

> 精度敏感的部分（损失计算、归一化层、动作输出）通常保持 fp32。openpi 的 `to_bfloat16_for_selected_params` 就是在选择哪些参数保持 fp32。

---

## 5. 调试三板斧

1. **过拟合一个 batch**：同一个 batch 反复训练几百步，loss 应该接近 0。降不下去，说明模型或数据流有 bug。
2. **检查初始 loss**：例如 $K$ 类分类，随机初始化时 loss 应约为 $\log K$。差很多说明有问题。
3. **打印/可视化数据**：送进模型前，把图像画出来、把动作的统计量打出来。"数据错了"是机器人学习里最常见的 bug，比如左右相机装反、动作单位不对、时间没对齐。

### 练习 3：写一个可复用的训练函数，并用"过拟合一个 batch"做检查

把前面学的所有内容组合成一个函数：`train(model, loader, steps, lr)`，内部使用 AdamW、warmup+余弦调度（可用 `torch.optim.lr_scheduler.LambdaLR`）、梯度裁剪、EMA，并返回 loss 历史和 EMA 模型。

In [ ]:
import math

def train(model, loader, steps, lr=1e-3, warmup=50, clip=1.0, ema_decay=0.99, loss_fn=None, device="cpu"):
    """loss_fn(model, batch) -> 标量 loss。返回 (losses 列表, ema 对象)"""
    model.to(device).train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / warmup) * 0.5 * (1 + math.cos(math.pi * min(s / steps, 1.0))))
    ema = EMA(model, ema_decay)
    losses, it = [], iter(loader)
    for step in range(steps):
        try:
            batch = next(it)
        except StopIteration:            # 一个 epoch 结束，重新开始
            it = iter(loader); batch = next(it)
        batch = {k: v.to(device) for k, v in batch.items()}
        # TODO: 前向算 loss -> zero_grad -> backward -> clip_grad_norm_ -> opt.step -> sched.step -> ema.update
        raise NotImplementedError
    return losses, ema

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
import math

def train(model, loader, steps, lr=1e-3, warmup=50, clip=1.0, ema_decay=0.99, loss_fn=None, device="cpu"):
    """loss_fn(model, batch) -> 标量 loss。返回 (losses 列表, ema 对象)"""
    model.to(device).train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / warmup) * 0.5 * (1 + math.cos(math.pi * min(s / steps, 1.0))))
    ema = EMA(model, ema_decay)
    losses, it = [], iter(loader)
    for step in range(steps):
        try:
            batch = next(it)
        except StopIteration:
            it = iter(loader); batch = next(it)
        batch = {k: v.to(device) for k, v in batch.items()}
        loss = loss_fn(model, batch)
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        opt.step()
        sched.step()
        ema.update(model)
        losses.append(loss.item())
    return losses, ema

In [ ]:
# ✅ 自测：先"过拟合一个 batch"，再正常训练
H = 8
policy = nn.Sequential(nn.Linear(4, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, H * 2))

def chunk_loss(model, batch):
    pred = model(batch["obs"]).view(-1, H, 2)                       # (B, H, 2)
    err = ((pred - batch["actions"]) ** 2).mean(-1)                  # (B, H)
    return (err * batch["mask"]).sum() / batch["mask"].sum()          # 只在真实动作上算 loss

one_batch = [next(iter(loader))]                                     # 只有一个 batch 的"数据集"
losses, _ = train(policy, one_batch, steps=300, lr=3e-3, loss_fn=chunk_loss)
print(f"过拟合单个 batch: loss {losses[0]:.4f} -> {losses[-1]:.5f}")
assert losses[-1] < 0.1 * losses[0]

policy = nn.Sequential(nn.Linear(4, 128), nn.ReLU(), nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, H * 2))
losses, ema = train(policy, loader, steps=1500, lr=3e-3, loss_fn=chunk_loss)
print(f"正常训练: loss {np.mean(losses[:50]):.4f} -> {np.mean(losses[-50:]):.4f}")
print("通过 ✅")

最后，用训练好的（EMA）策略在环境里跑一下。**动作块策略每次预测 H 步，执行前几步后再重新预测**（这叫 receding horizon，滚动时域执行）：

In [ ]:
@torch.no_grad()
def evaluate(policy, n_episodes=100, exec_steps=4, seed=123):
    env = PointReachEnv(num_envs=n_episodes, seed=seed)
    obs = env.reset()
    while not env.done.all():
        chunk = policy(torch.as_tensor(obs)).view(-1, H, 2).numpy()   # (N, H, 2)
        for k in range(exec_steps):                                    # 执行前 exec_steps 步
            obs, _, done, info = env.step(chunk[:, k])
    return info["success"].mean()

print("EMA 策略成功率:", evaluate(ema.shadow))

---

## 小结：训练脚本清单

- [ ] `Dataset.__getitem__` 返回字典；轨迹数据要处理"帧索引 → 动作块"与 padding mask
- [ ] `DataLoader(shuffle=True, drop_last=True, num_workers=...)`
- [ ] AdamW + warmup/cosine + 梯度裁剪
- [ ] EMA 影子模型用于评估与部署
- [ ] 存档包含 model / optimizer / step；加载时检查 missing/unexpected keys
- [ ] 推理前 `model.eval()` + `torch.no_grad()`
- [ ] 调试：过拟合单 batch、检查初始 loss、可视化数据

🎉 **第 1 章完成！** 你已经拥有了阅读和编写深度学习代码的全部基础。下一章进入强化学习：[第 2 章](../02_rl/README.md)。